# Loan Default Scorecard — Exploratory Data Analysis
**Dataset:** Give Me Some Credit (Kaggle)  
**Target:** `SeriousDlqin2yrs` — 1 if applicant defaulted within 2 years

---

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from utils import load_raw_data, FEATURES, TARGET

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

df = load_raw_data('cs-training.csv')
print(f'Shape: {df.shape}')
df.head()

## 1. Target Distribution

In [ ]:
dr = df[TARGET].mean()
print(f'Default rate: {dr:.2%}')
print(f'Class imbalance ratio: {(1-dr)/dr:.1f}:1 (Good:Bad)\n')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[TARGET].value_counts().plot.bar(ax=axes[0], color=['#27ae60','#e74c3c'], edgecolor='white')
axes[0].set_xticklabels(['No Default (0)', 'Default (1)'], rotation=0)
axes[0].set_title('Target Distribution', fontweight='bold')

axes[1].pie([1-dr, dr], labels=['Good', 'Bad'], colors=['#27ae60','#e74c3c'],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Good / Bad Split', fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
miss_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
miss_df = miss_df[miss_df['Missing Count'] > 0]
print(miss_df)

if len(miss_df) > 0:
    miss_df['Missing %'].plot.barh(color='#e74c3c', edgecolor='white', figsize=(8, 3))
    plt.title('Missing Values by Feature (%)', fontweight='bold')
    plt.xlabel('% Missing')
    plt.tight_layout()
    plt.show()

## 3. Feature Distributions

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, feat in enumerate(FEATURES):
    ax = axes[i]
    data = df[feat].dropna()
    # Cap extreme values for display
    p99 = data.quantile(0.99)
    data_capped = data.clip(upper=p99)
    ax.hist(data_capped, bins=40, color='#2c3e50', edgecolor='white', alpha=0.8)
    ax.set_title(feat, fontsize=10, fontweight='bold')
    ax.set_ylabel('Count')

for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Feature Distributions (capped at 99th percentile)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Default Rate by Feature (Univariate Analysis)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, feat in enumerate(FEATURES):
    ax = axes[i]
    df_temp = df[[feat, TARGET]].dropna()
    try:
        df_temp['bin'] = pd.qcut(df_temp[feat], q=10, duplicates='drop')
        dr_by_bin = df_temp.groupby('bin')[TARGET].mean()
        dr_by_bin.plot.bar(ax=ax, color='#e74c3c', edgecolor='white', alpha=0.8)
        ax.set_title(f'{feat}', fontsize=9, fontweight='bold')
        ax.set_ylabel('Default Rate')
        ax.tick_params(axis='x', labelsize=6, rotation=45)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    except Exception:
        ax.text(0.5, 0.5, 'Insufficient data', transform=ax.transAxes, ha='center')

for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Default Rate by Feature Decile', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Correlation Matrix

In [ ]:
corr_cols = FEATURES + [TARGET]
corr = df[corr_cols].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Age & Income Profiles by Default Status

In [ ]:
df_imp = df.copy()
df_imp['monthly_income'].fillna(df_imp['monthly_income'].median(), inplace=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for status, color, label in [(0, '#27ae60', 'Good'), (1, '#e74c3c', 'Bad (Default)')]:
    sub = df_imp[df_imp[TARGET] == status]
    axes[0].hist(sub['age'], bins=30, alpha=0.6, color=color, label=label, edgecolor='white', density=True)
    income_capped = sub['monthly_income'].clip(upper=sub['monthly_income'].quantile(0.99))
    axes[1].hist(income_capped, bins=30, alpha=0.6, color=color, label=label, edgecolor='white', density=True)

axes[0].set_title('Age Distribution by Default Status', fontweight='bold')
axes[0].set_xlabel('Age'); axes[0].legend()
axes[1].set_title('Monthly Income by Default Status', fontweight='bold')
axes[1].set_xlabel('Monthly Income'); axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Summary Statistics

In [ ]:
print('=== Summary Statistics by Default Status ===\n')
summary = df.groupby(TARGET)[FEATURES].median().T
summary.columns = ['Good (0)', 'Bad (1)']
summary['Ratio (Bad/Good)'] = (summary['Bad (1)'] / summary['Good (0)']).round(2)
print(summary.to_string())